In [1]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
import os
from google.colab import files

# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = ""

In [ ]:
# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# ========== GLOBAL CONFIG ==========
PLAYLIST_FILE = os.path.join(DATA_PATH, "playlist_final_unweighted.csv")
TRACKS_FILE = os.path.join(DATA_PATH, "track_final.parquet")
PLAYLIST_RECOMMENDED = os.path.join(DATA_PATH, "recommendations_for_all_models.csv")
NUM_PLAYLISTS = 1000000
K_EVAL = 50

Using device: cuda


In [5]:
playlist = pd.read_csv(PLAYLIST_FILE, engine='python', on_bad_lines='skip')
playlist_with_recommendation = pd.read_csv(PLAYLIST_RECOMMENDED, engine='python', on_bad_lines='skip')
track = pd.read_parquet(TRACKS_FILE)
recommendations_for_all_models = playlist_with_recommendation.copy()

This notebook only contains the code of the hybrid model. More analysis will be done in `step_7b_hybrid_model_basic_analysis.ipynb`.

# Collaborative Filtering using SVD

In [6]:
import pandas as pd
import numpy as np
import re
from sklearn.decomposition import TruncatedSVD
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

### Model Loading & Pre-Processing

In [7]:
# Only adjust the CLUSTER, WEIGHT_SVD, WEIGHT_DNN, default_weights, DATA_PATH, PLAYLIST_FILE, TRACKS_FILE

# ========== GLOBAL CONFIGURATION ==========
CLUSTER = None
WEIGHT_SVD = 0.5
WEIGHT_DNN = 0.5

# ========== SVD ==========
LATENT_DIM = 100

# ========== DNN ==========
# Default Feature Weights
default_weights = {
    'popularity': 0.2,
    'era': 0.2,
    'length': 0.2,
    'sentiment': 0.2,
    'genre': 0.2
}
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Model Architecture
INPUT_SIZE = None  # will be determined later based on the dataset
HIDDEN_SIZES = [512, 256, 128]
DROPOUT_RATE = 0.1
ACTIVATION = nn.ReLU

# Training Parameters
BATCH_SIZE = 512
EPOCHS = 30
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
LR_STEP_SIZE = 5
LR_GAMMA = 0.5
NEGATIVE_SAMPLE_RATIO = 10  # n_neg in PlaylistTrackDataset

# Loss multiplier
POS_WEIGHT_MULTIPLIER = 1.5

Using device: cuda


In [8]:
###################################
# 1. DATA LOADING & PREPROCESSING (SVD)
###################################
def load_data():
    playlists = pd.read_csv(PLAYLIST_FILE, engine='python', on_bad_lines='skip')
    tracks = pd.read_parquet(TRACKS_FILE)
    if CLUSTER:
        playlists = playlists[playlists['cluster'] == CLUSTER].reset_index(drop=True)[:NUM_PLAYLISTS]
    else:
        playlists = playlists[:NUM_PLAYLISTS]
    return playlists, tracks

def clean_centroid_string(s):
    """Converts a centroid string into a list of floats."""
    if isinstance(s, str):
        s = re.sub(r'[\[\]]', '', s).strip()
        return [float(x) for x in s.split()]
    return s

def parse_track_list(s):
    """Extracts integers from a string and returns them as a list."""
    return [int(x) for x in re.findall(r'\d+', s)]

def preprocess_playlists(df):
    """Preprocesses the playlist DataFrame by cleaning centroids and parsing track lists."""
    df = df.copy()
    for col in ['sentiment_centroid', 'genre_centroid']:
        if col in df.columns:
            df[col] = df[col].apply(clean_centroid_string)
    for col in ['track_idx_list', 'tracks_to_predict']:
        if col in df.columns:
            df[col] = df[col].apply(parse_track_list)
    return df

In [9]:
###################################
# 2. DATA SPLITTING & INTERACTION MATRIX (SVD)
###################################
def split_data(playlists):
    """
    Splits playlists into train, validation, and test sets
    based on the 'dataset_type' column.
    """
    train = playlists[playlists['dataset_type'].isin(['train', 'val'])].reset_index(drop=True)
    val = playlists[playlists['dataset_type'] == 'val'].reset_index(drop=True)
    test = playlists[playlists['dataset_type'] == 'test'].reset_index(drop=True)
    final = playlists[playlists['dataset_type'] == 'final'].reset_index(drop=True)
    return train, val, test, final

def build_track_mapping(tracks_df):
    """Creates a sorted list of all available tracks and a mapping to column indices."""
    unique_tracks = sorted(tracks_df['track_idx'].unique().tolist())
    track_to_col = {track: idx for idx, track in enumerate(unique_tracks)}
    return unique_tracks, track_to_col

def build_interaction_matrix(playlists_subset, track_to_col, track_col='track_idx_list'):
    """Builds a binary interaction matrix for the given subset of playlists."""
    n = len(playlists_subset)
    m = len(track_to_col)
    matrix = np.zeros((n, m), dtype=np.int32)
    for i, row in playlists_subset.iterrows():
        for track in row[track_col]:
            if track in track_to_col:
                matrix[i, track_to_col[track]] = 1
    return matrix

def create_track_feat_map_svd(tracks_df):
    """
    Creates a feature mapping for tracks using the full tracks DataFrame.
    """
    feat_cols = ['track_popularity',
                 'Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era',
                 'Short', 'Medium', 'Long',
                 'joy', 'calm', 'sadness', 'fear', 'energizing', 'dreamy',
                 'Instrumental / Ambient Sounds', 'Soft Acoustic / Classical', 'Orchestral / Soundtrack',
                 'Mid-tempo Pop / Indie', 'Upbeat Electronic / Dance', 'Slow & Melancholic (Sad Songs)',
                 'Experimental / Jazz Fusion', 'Lo-Fi / Chill Vibes']
    feat_map = {row['track_idx']: row[feat_cols].values for _, row in tracks_df.iterrows()}
    return feat_map

In [10]:
###################################
# 3. SVD MODEL TRAINING, PREDICTION & EVALUATION (SVD)
###################################
def train_svd_model(interaction_matrix_train):
    """Trains an SVD model on the training interaction matrix."""
    svd = TruncatedSVD(n_components=LATENT_DIM, random_state=42)
    U_train = svd.fit_transform(interaction_matrix_train)
    Sigma = svd.singular_values_
    VT = svd.components_
    sqrt_sigma = np.sqrt(Sigma)
    P_train = U_train * sqrt_sigma  # Playlist latent factors
    Q = (VT.T * sqrt_sigma)         # Track latent factors
    return svd, P_train, Q, sqrt_sigma

def fold_in_playlists(svd, interaction_matrix, sqrt_sigma):
    """Folds in new playlists into the SVD latent space."""
    return svd.transform(interaction_matrix) * sqrt_sigma

def predict_mf(playlist_latent, interaction_row, Q):
    """
    Predicts scores for all tracks for a given playlist using its latent factors.
    Existing tracks are excluded by setting their scores to -∞.
    """
    scores = playlist_latent.dot(Q.T)
    existing = np.where(interaction_row > 0)[0]
    scores[existing] = -np.inf
    return scores

def predict_mf_wrapper(playlist_idx, interaction_matrix, P_matrix, Q):
    """Wrapper to predict scores for a playlist by its index."""
    return predict_mf(P_matrix[playlist_idx], interaction_matrix[playlist_idx], Q)

def get_all_svd_predictions(interaction_matrix, P_matrix, predict_func, Q, test_playlists):
    """
    Returns a dictionary mapping the original playlist IDs (from test_playlists)
    to the predicted score vector for each playlist.
    """
    predictions = {}
    n = interaction_matrix.shape[0]
    for i in range(n):
        # Get the original playlist ID
        playlist_id = test_playlists.iloc[i]['playlist_idx']
        predictions[playlist_id] = predict_func(i, interaction_matrix, P_matrix, Q)
    return predictions

def compute_metrics_for_playlist(predicted_scores, test_indices, k=10):
    """
    Computes Hit@k, MRR, and MAP for a given playlist.
    """
    ranked_indices = np.argsort(-predicted_scores)
    top_k = ranked_indices[:k]
    hit = 1 if any(t in top_k for t in test_indices) else 0
    precisions = []
    num_hits = 0
    mrr = 0.0
    for rank_idx, track_idx in enumerate(ranked_indices[:k]):
        if track_idx in test_indices:
            num_hits += 1
            precisions.append(num_hits / (rank_idx + 1))
            if mrr == 0.0:
                mrr = 1.0 / (rank_idx + 1)
    ap = np.mean(precisions) if precisions else 0.0
    return hit, mrr, ap

def build_ground_truth(playlists_subset, track_to_col, track_col='tracks_to_predict'):
    """
    Builds a dictionary mapping each playlist (using its original 'playlist_idx')
    to a list of ground truth track indices.
    """
    test_items = {}
    for _, row in playlists_subset.iterrows():
        pid = row['playlist_idx']
        test_items[pid] = [track_to_col[t] for t in row[track_col] if t in track_to_col]
    return test_items

def evaluate_model_svd(interaction_matrix, ground_truth, P_matrix, predict_func, Q, playlistid_to_index, k=50):
    """
    Evaluates the SVD model using ground truth keyed by original playlist IDs.
    It returns only Hit@K, MRR, and MAP@K.
    """
    hit_total, mrr_total, ap_total = 0, 0, 0
    n = len(ground_truth)
    for pid, true_indices in ground_truth.items():
        if pid not in playlistid_to_index:
            continue  # skip if the playlist ID is not found in the test data
        idx = playlistid_to_index[pid]
        predicted_scores = predict_func(idx, interaction_matrix, P_matrix, Q)
        hit, mrr, ap = compute_metrics_for_playlist(predicted_scores, true_indices, k)
        hit_total += hit
        mrr_total += mrr
        ap_total += ap
    return {
        'Hit@K': hit_total / n,
        'MRR': mrr_total / n,
        'MAP@K': ap_total / n
    }

### Training

In [11]:
# Load and preprocess data
playlist_raw, tracks = load_data()
playlists = preprocess_playlists(playlist_raw)

# Split data into train, validation, test, and final sets
train_playlists, val_playlists, test_playlists, final_playlists = split_data(playlists)

# Build track mapping (based on training data) and interaction matrices
unique_tracks, track_to_col = build_track_mapping(tracks)
print("Training playlists:", len(train_playlists))
print("Unique tracks (from train):", len(unique_tracks))

interaction_matrix_train = build_interaction_matrix(train_playlists, track_to_col)
interaction_matrix_val = build_interaction_matrix(val_playlists, track_to_col)
interaction_matrix_test = build_interaction_matrix(test_playlists, track_to_col)
interaction_matrix_final = build_interaction_matrix(final_playlists, track_to_col)
print("Training interaction matrix shape:", interaction_matrix_train.shape)
print("Validation interaction matrix shape:", interaction_matrix_val.shape)
print("Test interaction matrix shape:", interaction_matrix_test.shape)
print("Final interaction matrix shape:", interaction_matrix_final.shape)

track_feat_map = create_track_feat_map_svd(tracks)

# Train SVD model on training set
svd, P_train, Q, sqrt_sigma = train_svd_model(interaction_matrix_train)

Training playlists: 13877
Unique tracks (from train): 252236
Training interaction matrix shape: (13877, 252236)
Validation interaction matrix shape: (1850, 252236)
Test interaction matrix shape: (1851, 252236)
Final interaction matrix shape: (2776, 252236)


### Generate Results for Test Set

In [12]:
# Fold in test playlists
P_test = fold_in_playlists(svd, interaction_matrix_test, sqrt_sigma)
# Build ground truth for test
ground_truth_test = build_ground_truth(test_playlists, track_to_col)
# Build mapping from original playlist IDs to row indices for the test set
playlistid_to_index_test = {row['playlist_idx']: i for i, row in test_playlists.iterrows()}

# Convert test recommendations to DataFrame
index_to_track = {v: k for k, v in track_to_col.items()}
test_recommendations = []
n_test = interaction_matrix_test.shape[0]
for i in range(n_test):
    scores = predict_mf_wrapper(i, interaction_matrix_test, P_test, Q)
    ranked_indices = np.argsort(-scores)
    top_indices = ranked_indices[:K_EVAL]
    recommended_tracks = [index_to_track[idx] for idx in top_indices if idx in index_to_track]
    playlist_id = test_playlists.iloc[i]['playlist_idx']
    test_recommendations.append({'playlist_idx': playlist_id, 'recommended_tracks': recommended_tracks})
svd_rec_df = pd.DataFrame(test_recommendations)

# Content Based Filtering using DNN

### Model Loading & Pre-Processing

In [13]:
###################################
# 1. DATA LOADING & PREPROCESSING (DNN)
###################################
def preprocess_playlist(playlist):
    # Convert centroids from strings to numpy arrays and unpack them
    playlist['sentiment_centroid'] = playlist['sentiment_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
    playlist['genre_centroid'] = playlist['genre_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

    sent_df = playlist['sentiment_centroid'].apply(pd.Series)
    sent_df.columns = [f'sent{i+1}' for i in range(sent_df.shape[1])]
    genre_df = playlist['genre_centroid'].apply(pd.Series)
    genre_df.columns = [f'genre{i+1}' for i in range(genre_df.shape[1])]

    playlist = pd.concat([playlist.drop(columns=['sentiment_centroid', 'genre_centroid']), sent_df, genre_df], axis=1)

    def convert_string_array_to_list(s):
        if isinstance(s, str):
            return [int(x) for x in re.findall(r'\d+', s)]
        return []

    playlist['track_idx_list'] = playlist['track_idx_list'].apply(convert_string_array_to_list)
    playlist['tracks_to_predict'] = playlist['tracks_to_predict'].apply(convert_string_array_to_list)

    cols = ['playlist_idx', 'dataset_type', 'track_idx_list', 'tracks_to_predict', 'cluster',
            'popularity_mean', 'era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion',
            'era_2000s_proportion', 'era_modern_era_proportion', 'length_short_proportion', 'length_medium_proportion',
            'length_long_proportion', 'sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6',
            'genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']
    return playlist[cols]

# Load and preprocess
playlist_raw, tracks = load_data()
playlist = preprocess_playlist(playlist_raw)
train_playlists_dnn, val_playlists_dnn, test_playlists_dnn, final_playlists_dnn = split_data(playlist)

# Define playlist feature columns and create scalers
playlist_popularity_cols = ['popularity_mean']
playlist_era_cols = ['era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion',
                       'era_2000s_proportion', 'era_modern_era_proportion']
playlist_length_cols = ['length_short_proportion', 'length_medium_proportion', 'length_long_proportion']
playlist_sentiment_cols = ['sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6']
playlist_genre_cols = ['genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']

playlist_scalers = {
    'pop_playlist': StandardScaler(),
    'era_playlist': StandardScaler(),
    'len_playlist': StandardScaler(),
    'sent_playlist': StandardScaler(),
    'genre_playlist': StandardScaler()
}

playlist_scalers['pop_playlist'].fit(train_playlists_dnn[playlist_popularity_cols])
playlist_scalers['era_playlist'].fit(train_playlists_dnn[playlist_era_cols])
playlist_scalers['len_playlist'].fit(train_playlists_dnn[playlist_length_cols])
playlist_scalers['sent_playlist'].fit(train_playlists_dnn[playlist_sentiment_cols])
playlist_scalers['genre_playlist'].fit(train_playlists_dnn[playlist_genre_cols])

def normalize_playlist_features(df):
    df[playlist_popularity_cols] = playlist_scalers['pop_playlist'].transform(df[playlist_popularity_cols])
    df[playlist_era_cols] = playlist_scalers['era_playlist'].transform(df[playlist_era_cols])
    df[playlist_length_cols] = playlist_scalers['len_playlist'].transform(df[playlist_length_cols])
    df[playlist_sentiment_cols] = playlist_scalers['sent_playlist'].transform(df[playlist_sentiment_cols])
    df[playlist_genre_cols] = playlist_scalers['genre_playlist'].transform(df[playlist_genre_cols])
    return df

train_playlists_dnn = normalize_playlist_features(train_playlists_dnn.copy())
val_playlists_dnn = normalize_playlist_features(val_playlists_dnn.copy())
test_playlists_dnn = normalize_playlist_features(test_playlists_dnn.copy())

def create_playlist_feat_map(df):
    feat_cols = playlist_popularity_cols + playlist_era_cols + playlist_length_cols + playlist_sentiment_cols + playlist_genre_cols
    feat_map = {row['playlist_idx']: row[feat_cols].values for _, row in df.iterrows()}
    return feat_map

playlist_feat_map_train = create_playlist_feat_map(train_playlists_dnn)
playlist_feat_map_val = create_playlist_feat_map(val_playlists_dnn)
playlist_feat_map_test = create_playlist_feat_map(test_playlists_dnn)

# For tracks in DNN, define feature columns and scalers
popularity_cols = ['track_popularity']
era_cols = ['Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era']
length_cols = ['Short', 'Medium', 'Long']
sentiment_cols = ['joy', 'calm', 'sadness', 'fear', 'energizing', 'dreamy']
genre_cols = ['Instrumental / Ambient Sounds', 'Soft Acoustic / Classical', 'Orchestral / Soundtrack',
              'Mid-tempo Pop / Indie', 'Upbeat Electronic / Dance', 'Slow & Melancholic (Sad Songs)',
              'Experimental / Jazz Fusion', 'Lo-Fi / Chill Vibes']

track_scalers = {
    'pop_track': StandardScaler(),
    'era_track': StandardScaler(),
    'len_track': StandardScaler(),
    'sent_track': StandardScaler(),
    'genre_track': StandardScaler()
}

track_scalers['pop_track'].fit(tracks[popularity_cols])
track_scalers['era_track'].fit(tracks[era_cols])
track_scalers['len_track'].fit(tracks[length_cols])
track_scalers['sent_track'].fit(tracks[sentiment_cols])
track_scalers['genre_track'].fit(tracks[genre_cols])

def normalize_track_features(df):
    df[popularity_cols] = track_scalers['pop_track'].transform(df[popularity_cols])
    df[era_cols] = track_scalers['era_track'].transform(df[era_cols])
    df[length_cols] = track_scalers['len_track'].transform(df[length_cols])
    df[sentiment_cols] = track_scalers['sent_track'].transform(df[sentiment_cols])
    df[genre_cols] = track_scalers['genre_track'].transform(df[genre_cols])
    return df

tracks = normalize_track_features(tracks.copy())

def create_track_feat_map(df):
    feat_cols = popularity_cols + era_cols + length_cols + sentiment_cols + genre_cols
    feat_map = {int(row['track_idx']): row[feat_cols].values for _, row in df.iterrows()}
    return feat_map

track_feat_map = create_track_feat_map(tracks)

def apply_weights(features, weights):
    split_sizes = [1, 5, 3, 6, 8]
    chunks = np.split(features, np.cumsum(split_sizes)[:-1])
    return np.concatenate([chunk * weights[key] for chunk, key in zip(chunks, weights)])

In [14]:
###################################
# 2. MODEL & DATASET DEFINITION (DNN)
###################################
class PlaylistTrackDataset(Dataset):
    def __init__(self, playlist_df, feat_map, track_map, n_neg=NEGATIVE_SAMPLE_RATIO, cluster_weights=None):
        self.samples = []
        self.track_map = track_map
        all_tids = list(track_map.keys())
        cluster_weights = cluster_weights or {}
        for _, row in playlist_df.iterrows():
            pid, cluster = row['playlist_idx'], row['cluster']
            pos_tracks = row['tracks_to_predict']
            if not pos_tracks:  # skip if no positive tracks
                continue
            p_feat = feat_map[pid]
            weights = cluster_weights.get(cluster, default_weights)
            p_feat_w = apply_weights(p_feat, weights)
            for tid in pos_tracks:
                if tid in track_map:
                    self.samples.append((p_feat_w, track_map[tid], 1))
            negs = np.random.choice(list(set(all_tids) - set(pos_tracks)),
                                    min(len(pos_tracks) * n_neg, len(all_tids)), replace=False)
            for tid in negs:
                self.samples.append((p_feat_w, track_map[tid], 0))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        p, t, y = self.samples[idx]
        p = np.array(p, dtype=np.float32).flatten()
        t = np.array(t, dtype=np.float32).flatten()
        return torch.tensor(np.concatenate([p, t]), dtype=torch.float32), torch.tensor([y], dtype=torch.float32)

class DNNRecommender(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        layers = []
        prev_size = input_size
        for hidden in HIDDEN_SIZES:
            layers.append(nn.Linear(prev_size, hidden))
            layers.append(ACTIVATION())
            layers.append(nn.BatchNorm1d(hidden))
            layers.append(nn.Dropout(DROPOUT_RATE))
            prev_size = hidden
        layers.append(nn.Linear(prev_size, 1))
        self.model = nn.Sequential(*layers)
    def forward(self, x):
        return self.model(x).squeeze()

### Training

In [15]:
# Create playlist feature maps for training and validation
playlist_feat_map_train = create_playlist_feat_map(train_playlists_dnn)
playlist_feat_map_val = create_playlist_feat_map(val_playlists_dnn)

# Create the dataset and DataLoader for training
train_dataset = PlaylistTrackDataset(train_playlists_dnn, playlist_feat_map_train, track_feat_map)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model, optimizer, and scheduler
model = DNNRecommender(input_size=len(next(iter(train_dataset))[0])).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

# Compute pos_weight for loss function
pos = sum(1 for _, _, l in train_dataset.samples if l == 1)
neg = sum(1 for _, _, l in train_dataset.samples if l == 0)
pos_weight_val = (neg/pos * POS_WEIGHT_MULTIPLIER)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_val).to(device))

# Training loop
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb).view(-1), yb.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 3217.1945
Epoch 2, Loss: 2966.7758
Epoch 3, Loss: 2880.8652
Epoch 4, Loss: 2816.6817
Epoch 5, Loss: 2768.0953
Epoch 6, Loss: 2669.5445
Epoch 7, Loss: 2630.3912
Epoch 8, Loss: 2601.0525
Epoch 9, Loss: 2576.3985
Epoch 10, Loss: 2556.9550
Epoch 11, Loss: 2498.1874
Epoch 12, Loss: 2475.6571
Epoch 13, Loss: 2460.4486
Epoch 14, Loss: 2442.7566
Epoch 15, Loss: 2433.9663
Epoch 16, Loss: 2397.4131
Epoch 17, Loss: 2381.0962
Epoch 18, Loss: 2376.3633
Epoch 19, Loss: 2365.2731
Epoch 20, Loss: 2359.5960
Epoch 21, Loss: 2343.3764
Epoch 22, Loss: 2333.7546
Epoch 23, Loss: 2328.7756
Epoch 24, Loss: 2323.1323
Epoch 25, Loss: 2324.3758
Epoch 26, Loss: 2314.6158
Epoch 27, Loss: 2309.4481
Epoch 28, Loss: 2308.8143
Epoch 29, Loss: 2306.6434
Epoch 30, Loss: 2303.0363


### Generate Results for Test Set

In [16]:
def predict_dnn(playlist_idx, feat_map, track_feat_map):
    p_feat = feat_map[playlist_idx]
    p_feat_w = apply_weights(p_feat, default_weights)
    all_track_ids = list(track_feat_map.keys())
    p_feat_tensor = torch.tensor(np.array(p_feat_w, dtype=np.float32)).repeat(len(all_track_ids), 1).to(device)
    t_feats = np.array([track_feat_map[tid] for tid in all_track_ids], dtype=np.float32)
    t_tensor = torch.tensor(t_feats, dtype=torch.float32).to(device)
    inputs = torch.cat([p_feat_tensor, t_tensor], dim=1)
    model.eval()
    with torch.no_grad():
        scores = torch.sigmoid(model(inputs)).cpu().numpy().flatten()
    return scores

def get_all_dnn_predictions(feat_map, track_feat_map, test_playlists):
    predictions = {}
    for i, row in test_playlists.iterrows():
        playlist_id = row['playlist_idx']
        predictions[playlist_id] = predict_dnn(playlist_id, feat_map, track_feat_map)
    return predictions

def compute_metrics_for_playlist(predicted_tracks, true_tracks, k):
    top_k = predicted_tracks[:k]
    hit = int(any(t in top_k for t in true_tracks))
    precisions = []
    mrr = 0.0
    num_hits = 0
    for rank, track_id in enumerate(top_k):
        if track_id in true_tracks:
            num_hits += 1
            precisions.append(num_hits / (rank + 1))
            if mrr == 0.0:
                mrr = 1.0 / (rank + 1)
    ap = np.mean(precisions) if precisions else 0.0
    return hit, mrr, ap

def evaluate_model(model, test_playlists, playlist_feat_map, track_feat_map, k=K_EVAL, weights_by_cluster=None):
    model.eval()
    all_track_ids = list(track_feat_map.keys())
    total_hit, total_mrr, total_ap = 0, 0, 0
    n = len(test_playlists)
    with torch.no_grad():
        for _, row in test_playlists.iterrows():
            pid = row['playlist_idx']
            cluster = row['cluster']
            true_tracks = row['tracks_to_predict']
            seen_tracks = set(row['track_idx_list'])
            p_feat = playlist_feat_map[pid]
            weights = weights_by_cluster.get(cluster, default_weights) if weights_by_cluster else default_weights
            p_feat_w = apply_weights(p_feat, weights)
            p_feat_tensor = torch.tensor(np.array(p_feat_w, dtype=np.float32)).repeat(len(all_track_ids), 1)
            t_feats = np.array([track_feat_map[tid] for tid in all_track_ids], dtype=np.float32)
            t_tensor = torch.tensor(t_feats, dtype=torch.float32)
            inputs = torch.cat([p_feat_tensor, t_tensor], dim=1)
            scores = torch.sigmoid(model(inputs.to(device))).cpu().numpy().flatten()
            ranked_tracks = [tid for tid, _ in sorted(zip(all_track_ids, scores), key=lambda x: x[1], reverse=True)]
            ranked_unseen = [tid for tid in ranked_tracks if tid not in seen_tracks]
            hit, mrr, ap = compute_metrics_for_playlist(ranked_unseen, true_tracks, k)
            total_hit += hit
            total_mrr += mrr
            total_ap += ap
    return {
        'Hit@K': total_hit / n,
        'MRR': total_mrr / n,
        'MAP@K': total_ap / n
    }

# Use test set for evaluation
playlist_feat_map_test = create_playlist_feat_map(test_playlists_dnn)

# Export recommendations from test set
test_recommendations = []
all_track_ids = list(track_feat_map.keys())
for _, row in test_playlists_dnn.iterrows():
    playlist_id = row['playlist_idx']
    scores = predict_dnn(playlist_id, playlist_feat_map_test, track_feat_map)
    ranked_indices = np.argsort(-scores)
    top_indices = ranked_indices[:K_EVAL]
    recommended_tracks = [all_track_ids[i] for i in top_indices if i < len(all_track_ids)]
    test_recommendations.append({'playlist_idx': playlist_id, 'recommended_tracks': recommended_tracks})
dnn_rec_df = pd.DataFrame(test_recommendations)
print(dnn_rec_df.head())

   playlist_idx                                 recommended_tracks
0             2  [26657, 169451, 130032, 135471, 72286, 58351, ...
1             7  [168149, 29246, 224295, 102680, 29052, 63251, ...
2            10  [234445, 247305, 203517, 37636, 208248, 29052,...
3            17  [237495, 123797, 221079, 208248, 149917, 15629...
4            28  [237495, 29052, 250306, 26659, 251276, 234445,...


# Hybrid Model (Content Based + Collaborative Filtering)

In [ ]:
# Get predictions on test using original playlist IDs as keys
svd_predictions = get_all_svd_predictions(interaction_matrix_test, P_test, predict_mf_wrapper, Q, test_playlists)
dnn_predictions = get_all_dnn_predictions(playlist_feat_map_test, track_feat_map, test_playlists_dnn)

# Ensure predictions from both models are aligned.
svd_track_ids = set(track_to_col.keys())
dnn_track_ids = set(track_feat_map.keys())
union_track_ids = sorted(svd_track_ids.union(dnn_track_ids))
union_track_to_index = {tid: idx for idx, tid in enumerate(union_track_ids)}
union_length = len(union_track_ids)

final_predictions = {}
all_playlist_ids = set(svd_predictions.keys()).union(set(dnn_predictions.keys()))
for pid in all_playlist_ids:
    final_score = np.zeros(union_length, dtype=np.float32)
    if pid in svd_predictions:
        svd_vec = svd_predictions[pid]
        for track, idx in track_to_col.items():
            union_idx = union_track_to_index.get(track)
            if union_idx is not None and idx < len(svd_vec):
                final_score[union_idx] += WEIGHT_SVD * svd_vec[idx]
    if pid in dnn_predictions:
        dnn_vec = dnn_predictions[pid]
        dnn_track_list = sorted(dnn_track_ids)
        dnn_map = {tid: idx for idx, tid in enumerate(dnn_track_list)}
        for track, dnn_idx in dnn_map.items():
            union_idx = union_track_to_index.get(track)
            if union_idx is not None and dnn_idx < len(dnn_vec):
                final_score[union_idx] += WEIGHT_DNN * dnn_vec[dnn_idx]
    final_predictions[pid] = final_score

def evaluate_final_predictions(final_preds, ground_truth, track_to_col, k=50):
    total_hit, total_mrr, total_ap = 0, 0, 0
    n = len(ground_truth)
    # Build a mapping from matrix indices to track IDs.
    index_to_track = {v: k for k, v in track_to_col.items()}
    for pid, true_indices in ground_truth.items():
        pred_scores = final_preds.get(pid)
        if pred_scores is None:
            continue
        ranked_indices = np.argsort(-pred_scores)
        top_k = ranked_indices[:k]
        # Compute Hit@K
        hit = 1 if any(t in true_indices for t in top_k) else 0
        total_hit += hit
        # Compute MRR and AP
        num_hits = 0
        precisions = []
        mrr = 0.0
        for rank, track_idx in enumerate(top_k):
            if track_idx in true_indices:
                num_hits += 1
                precisions.append(num_hits / (rank + 1))
                if mrr == 0:
                    mrr = 1 / (rank + 1)
        ap = np.mean(precisions) if precisions else 0.0
        total_ap += ap
        total_mrr += mrr
    return {
        'Hit@K': total_hit / n,
        'MRR': total_mrr / n,
        'MAP@K': total_ap / n
    }
playlist_clusters = {row['playlist_idx']: row['cluster'] for _, row in test_playlists.iterrows()}

# ===== CONVERT COMBINED RECOMMENDATIONS TO PANDAS =====
# Export overall combined model recommendations:
combined_recommendations = []
for pid, final_score in final_predictions.items():
    ranked_indices = np.argsort(-final_score)
    top_indices = ranked_indices[:K_EVAL]
    recommended_tracks = [union_track_ids[i] for i in top_indices if i < len(union_track_ids)]
    combined_recommendations.append({
        'playlist_idx': pid,
        'hybrid_recommendations': recommended_tracks
    })
combined_rec_df = pd.DataFrame(combined_recommendations)
recommendations_for_all_models = recommendations_for_all_models.merge(combined_rec_df, on='playlist_idx', how='inner')
combined_rec_df.head()

,playlist_idx,hybrid_recommendations
0,2,"[186858, 175642, 111887, 157693, 174997, 55648..."
1,7,"[191313, 111863, 163571, 216260, 89133, 237774..."
2,10,"[208248, 94251, 218307, 8706, 122853, 212881, ..."
3,17,"[235367, 216374, 237415, 64550, 54047, 14975, ..."
4,28,"[250306, 26659, 86790, 167186, 64550, 180201, ..."


# Save Recommendations

In [ ]:
from google.colab import drive
output_file_colab = os.path.join(DATA_PATH, "recommendations_for_all_models_with_hybrid.csv")

recommendations_for_all_models.to_csv(output_file_colab, index=False)
print(f"Recommendations saved to {output_file_colab} in Colab.")

Recommendations saved to /content/recommendations_for_all_models_with_hybrid.csv in Colab.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Mounted at /content/drive
